# Démonstration de `BinaryDecisionHead`

Ce notebook illustre le cycle complet `réponses angulaires -> agrégation -> calibration PyThresh -> décisions binaires`. Les données sont synthétiques afin de tester le composant indépendamment des filtres.

In [ ]:
import sys
from functools import partial
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (CURRENT_DIR, *CURRENT_DIR.parents) if (path / "src" / "scoring").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Impossible de trouver la racine du projet.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import matplotlib.pyplot as plt
import torch
from pythresh.thresholds.karch import KARCH

from scoring.aggregation import mean_score, max_score, top_quantile_mean_score
from scoring.decision import DecisionHead
from scoring.data_type import DecisionConfig

## Réponses angulaires synthétiques

Chaque patch possède huit scores angulaires. Les patchs normaux ont uniquement de faibles réponses ; les anomalies possèdent une réponse forte pour une orientation dominante. Les labels servent seulement à vérifier la démonstration et ne sont jamais transmis à `fit()`.

In [ ]:
def make_angular_responses(
    n_normal: int,
    n_anomalous: int,
    n_angles: int,
    seed: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    generator = torch.Generator().manual_seed(seed)

    normal = (
        0.15 + 0.04 * torch.randn((n_normal, n_angles), generator=generator)
    ).clamp_min(0)

    anomalous = (
        0.20 + 0.05 * torch.randn((n_anomalous, n_angles), generator=generator)
    ).clamp_min(0)
    rows = torch.arange(n_anomalous)
    dominant_angles = torch.randint(
        0, n_angles, (n_anomalous,), generator=generator
    )
    anomalous[rows, dominant_angles] += (
        0.8 + 0.08 * torch.randn(n_anomalous, generator=generator)
    )

    responses = torch.cat([normal, anomalous])
    labels = torch.cat([
        torch.zeros(n_normal, dtype=torch.bool),
        torch.ones(n_anomalous, dtype=torch.bool),
    ])
    order = torch.randperm(len(responses), generator=generator)
    return responses[order], labels[order]


calibration_responses, _ = make_angular_responses(100, 10, 8, seed=0)
test_responses, test_labels = make_angular_responses(24, 8, 8, seed=1)

print(f"Calibration: {tuple(calibration_responses.shape)}")
print(f"Test: {tuple(test_responses.shape)}")

## Calibration et prédiction

`max_score` garde la meilleure orientation de chaque patch. `KARCH.fit()` calibre ensuite son seuil uniquement à partir des scores du jeu de calibration.

In [ ]:
config = DecisionConfig(
    aggregator=max_score,
    threshold=KARCH(),
)
head = DecisionHead(config).fit(calibration_responses)

test_scores = head.score(test_responses)
test_predictions = head.predict(test_responses)

true_positives = (test_predictions & test_labels).sum().item()
false_positives = (test_predictions & ~test_labels).sum().item()
false_negatives = (~test_predictions & test_labels).sum().item()

print(f"Anomalies réelles : {test_labels.sum().item()}")
print(f"Anomalies détectées : {test_predictions.sum().item()}")
print(f"TP={true_positives}, FP={false_positives}, FN={false_negatives}")

In [ ]:
order = torch.argsort(test_scores)
sorted_scores = test_scores[order].detach().cpu()
sorted_predictions = test_predictions[order].detach().cpu()
sorted_labels = test_labels[order].detach().cpu()

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["tab:red" if prediction else "tab:blue" for prediction in sorted_predictions]
ax.scatter(range(len(sorted_scores)), sorted_scores, c=colors, label="Décision PyThresh")
ax.scatter(
    torch.where(sorted_labels)[0],
    sorted_scores[sorted_labels],
    marker="x",
    s=80,
    color="black",
    label="Anomalie synthétique réelle",
)
ax.set_xlabel("Patchs triés par score")
ax.set_ylabel("Score agrégé")
ax.set_title("Scores continus et décisions binaires")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Comparer les agrégateurs

Chaque essai utilise une nouvelle instance de `KARCH`, car un thresholder conserve l'état appris par `fit()`. Avec huit angles, un top 25 % moyenne les deux plus fortes réponses.

In [ ]:
aggregators = {
    "mean": mean_score,
    "max": max_score,
    "top_25_percent": partial(top_quantile_mean_score, fraction=0.25),
}

for name, aggregator in aggregators.items():
    current_head = DecisionHead(
        DecisionConfig(aggregator=aggregator, threshold=KARCH())
    ).fit(calibration_responses)

    predictions = current_head.predict(test_responses)
    tp = (predictions & test_labels).sum().item()
    fp = (predictions & ~test_labels).sum().item()
    fn = (~predictions & test_labels).sum().item()
    print(f"{name:16s} TP={tp:2d}  FP={fp:2d}  FN={fn:2d}")